In [1]:
import pandas as pd
import numpy as np

In [2]:
cartel_firms = pd.read_excel('../data/processed/clean_firm_data.xlsx')
listed_firms = pd.read_excel('../data/orbis_data/treat_listed_base.xlsx', sheet_name='Risultati',  dtype={'Codice NACE Rev. 2, core code (4 cifre)': str})
orbis_match_treated = pd.read_excel('../data/orbis_data/orbis_match_treated.xlsx')

# merge firm data with orbis match data 
cartel_firms = cartel_firms.merge(
    orbis_match_treated[['firm_name','bvd_id']],
    on = 'firm_name',
    how='left'
)

cartel_firms = cartel_firms.merge(
    listed_firms[['Numero BvD ID', 'Codice NACE Rev. 2, core code (4 cifre)']],
    left_on= 'bvd_id',
    right_on= 'Numero BvD ID',
    how='right'
)

In [3]:
cartel_firms['start_year'] = cartel_firms['start_date'].dt.year
cartel_firms['end_year'] = cartel_firms['end_date'].dt.year
firms_aft98 = cartel_firms[cartel_firms['start_year'] >= 1998]
firms_aft82 = cartel_firms[cartel_firms['start_year'] >= 1982]

# extract nace codes of treated quoted firms
nace_listed = cartel_firms[['Codice NACE Rev. 2, core code (4 cifre)']].drop_duplicates()
nace_listed.to_csv('../data/nace_listed.csv', sep="\t", index=False)
nace_listed_aft98 = firms_aft98[['Codice NACE Rev. 2, core code (4 cifre)']].drop_duplicates()
nace_listed_aft98.to_csv('../data/nace_listed_aft98.csv', sep="\t", index=False)
nace_listed_aft82 = firms_aft82[['Codice NACE Rev. 2, core code (4 cifre)']].drop_duplicates()
nace_listed_aft82.to_csv('../data/nace_listed_aft82.csv', sep="\t", index=False)

bvdid_listed = cartel_firms[['bvd_id']].drop_duplicates()
bvdid_listed_aft98 = firms_aft98[['bvd_id']].drop_duplicates()
bvdid_listed_aft82 = firms_aft82[['bvd_id']].drop_duplicates()
bvdid_listed_aft98.to_csv('../data/bvdid_listed_aft98.csv', sep="\t", index=False)
bvdid_listed_aft82.to_csv('../data/bvdid_listed_aft82.csv', sep="\t", index=False)

## data import

In [4]:
# firm data
firms_treated = pd.read_excel('../data/processed/clean_firm_data.xlsx')
# orbis
orbis_data = pd.read_csv('../data/processed/clean_orbis_base.csv', dtype={'nace_code': str})
orbis_match_treated = pd.read_excel('../data/orbis_data/orbis_match_treated.xlsx')
# regpat data
regpat_data = pd.read_csv('../data/processed/pct_clean.csv')
# match df
match_df = pd.read_csv('../data/firm_matches.csv')
treat_matches = pd.read_csv('../data/treat_matches.csv', sep="\t")

## data cleaning and prep

In [5]:
# drop extra rows from previous names
orbis_data = orbis_data.dropna(subset=['Unnamed: 0'])

# estract 2 digit nace code
orbis_data['nace_2dig'] = orbis_data['nace_code'].astype(str).str[:2]

In [6]:
# merge firm data with orbis match data 
firms_treated = firms_treated.merge(
    orbis_match_treated[['firm_name','bvd_id']],
    on = 'firm_name',
    how='left'
)
firms_treated['start_year'] = firms_treated['start_date'].dt.year
firms_treated['end_year'] = firms_treated['end_date'].dt.year
firms_treated['bvd_id'].nunique()

1247

## firm data processing

In [7]:
# create first treat df
first_cartel = firms_treated.groupby('bvd_id').agg(
    treat_year =('start_year', 'min'),
    end_year=('end_year', 'first'),
    cartel_id=('cartel_id', 'first')
).reset_index()

In [8]:
# merge cartel and orbis firm data
firms_df = pd.merge(
    orbis_data[['firm_id','firm_name_preproc', 'incorp_year', 'ctry_code', 'nace_code', 'nace_2dig', 'bvd_id']],
    first_cartel,
    on=['bvd_id'],
    how='left'
)

# create treatment and firm age indicators
firms_df['treated'] = firms_df['treat_year'].notna().astype(int)
firms_df['firm_age'] = pd.Timestamp.now().year - firms_df['incorp_year']
firms_df

,firm_id,firm_name_preproc,incorp_year,ctry_code,nace_code,nace_2dig,bvd_id,treat_year,end_year,cartel_id,treated,firm_age
0,0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
1,1,chinapetroleum&chemicalcorp,2000.0,CN,1920,19,CN30086PC,NaN,NaN,NaN,0,25.0
2,2,petrochinacolimited,1999.0,CN,1920,19,CN30081PC,NaN,NaN,NaN,0,26.0
3,3,appleinc,1977.0,US,2620,26,US942404110,NaN,NaN,NaN,0,48.0
4,4,volkswagenag,1937.0,DE,2910,29,DE2070000543,2002.0,2017.0,AT.40669,1,88.0
...,...,...,...,...,...,...,...,...,...,...,...,...
30361,30349,euroholdingsltd,NaN,MH,5222,52,MH*S00459082,NaN,NaN,NaN,0,NaN
30362,30350,syntheiacorp,2024.0,CA,6201,62,CA36329NC,NaN,NaN,NaN,0,1.0
30363,30351,commodoremetalscorp,2024.0,CA,0729,07,CA36581NC,NaN,NaN,NaN,0,1.0
30364,30352,medpalaiplc,2021.0,GB,6201,62,GB13578804,NaN,NaN,NaN,0,4.0


## patent data processing

In [9]:
# merge pat data with firm match df
regpat_data = pd.merge(
    regpat_data,
    match_df[['app_name_preproc', 'firm_id']],
    on='app_name_preproc',
    how='left'
)
regpat_data['year'] = pd.to_datetime(regpat_data['app_year'], format='%Y', errors='coerce').dt.year


In [10]:
# count patent fidings by firm each year
pat_year = (
    regpat_data
    .groupby(['firm_id', 'year'])
    .size()
    .reset_index(name='filed_pat')
)
pat_year.head()

,firm_id,year,filed_pat
0,1.0,2000.0,28
1,1.0,2001.0,90
2,1.0,2002.0,51
3,1.0,2003.0,105
4,1.0,2004.0,84


## panel

In [11]:
# create panel structure
years = range(1966, 2024)
panel_df = (
    firms_df['firm_id']
    .to_frame()
    .assign(key=1)
    .merge(pd.DataFrame({'year': years, 'key':1}), on='key')
    .drop(columns='key')
)

### merge with pat data

In [12]:
panel_df = panel_df.merge(pat_year, on=['firm_id','year'], how='left')
panel_df['filed_pat'] = panel_df['filed_pat'].fillna(0)

### merge with firm data

In [13]:
panel_df = panel_df.merge(firms_df, on='firm_id', how='left')
panel_df

,firm_id,year,filed_pat,firm_name_preproc,incorp_year,ctry_code,nace_code,nace_2dig,bvd_id,treat_year,end_year,cartel_id,treated,firm_age
0,0,1966,0.0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
1,0,1967,0.0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
2,0,1968,0.0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
3,0,1969,0.0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
4,0,1970,0.0,saudiarabianoilcosaudijointstockcompany,1988.0,SA,0610,06,SA30947GS,NaN,NaN,NaN,0,37.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1762615,30353,2019,0.0,miraeassetdreamspecialpurposeacquisition1company,2022.0,KR,6619,66,KR1101118406781,NaN,NaN,NaN,0,3.0
1762616,30353,2020,0.0,miraeassetdreamspecialpurposeacquisition1company,2022.0,KR,6619,66,KR1101118406781,NaN,NaN,NaN,0,3.0
1762617,30353,2021,0.0,miraeassetdreamspecialpurposeacquisition1company,2022.0,KR,6619,66,KR1101118406781,NaN,NaN,NaN,0,3.0
1762618,30353,2022,0.0,miraeassetdreamspecialpurposeacquisition1company,2022.0,KR,6619,66,KR1101118406781,NaN,NaN,NaN,0,3.0


## data overview and statistics

In [14]:
# summary of firms and treatment groups

tot_firms = panel_df['firm_id'].nunique()
num_treated_firms = panel_df.loc[panel_df['treated']==1, 'firm_id'].nunique()
num_control_firms = tot_firms - num_treated_firms
treated_share = num_treated_firms / tot_firms
num_cartels = panel_df['cartel_id'].nunique()
treated_firms_with_patent = panel_df[(panel_df['treated'] == 1) & (panel_df['filed_pat'] > 0)]
num_treated_firms_with_patent = treated_firms_with_patent['firm_id'].nunique()

print(f"total firms: {tot_firms}")
print(f"treated firms: {num_treated_firms}")
print(f"treated firms with at least one patent: {num_treated_firms_with_patent}")
print(f"control firms: {num_control_firms}")
print(f"treated share: {treated_share:.2f}%")
print(f"number of cartels: {num_cartels}")

total firms: 30354
treated firms: 196
treated firms with at least one patent: 127
control firms: 30158
treated share: 0.01%
number of cartels: 82


In [ ]:
tot_firms = pd.read_excel('../data/orbis_data/tot_firms/tot_firms.xlsx', sheet_name='Risultati')
panel_tot = panel_df.merge(tot_firms, left_on='bvd_id', right_on='Numero BvD ID', how='right')

# summary of firms and treatment groups
tot_firms = panel_tot['firm_id'].nunique()
num_treated_firms = panel_tot.loc[panel_tot['treated']==1, 'firm_id'].nunique()
num_control_firms = tot_firms - num_treated_firms
treated_share = num_treated_firms / tot_firms
num_cartels = panel_tot['cartel_id'].nunique()
treated_firms_with_patent = panel_tot[(panel_tot['treated'] == 1) & (panel_tot['filed_pat'] > 0)]
num_treated_firms_with_patent = treated_firms_with_patent['firm_id'].nunique()

print(f"total firms: {tot_firms}")
print(f"treated firms: {num_treated_firms}")
print(f"treated firms with at least one patent: {num_treated_firms_with_patent}")
print(f"control firms: {num_control_firms}")
print(f"treated share: {treated_share:.2f}%")
print(f"number of cartels: {num_cartels}")

total firms: 14641
treated firms: 169
treated firms with at least one patent: 112
control firms: 14472
treated share: 0.01%
number of cartels: 78


In [22]:
firms_aft98 = pd.read_excel('../data/orbis_data/tot_firms/tot_firms_aft98.xlsx',  sheet_name='Risultati')
panel_aft98 = panel_df.merge(firms_aft98, left_on='bvd_id', right_on='Numero BvD ID', how='right')

# summary of firms and treatment groups

tot_firms = panel_aft98['firm_id'].nunique()
num_treated_firms = panel_aft98.loc[panel_aft98['treated']==1, 'firm_id'].nunique()
num_control_firms = tot_firms - num_treated_firms
treated_share = num_treated_firms / tot_firms
num_cartels = panel_aft98['cartel_id'].nunique()
treated_firms_with_patent = panel_aft98[(panel_aft98['treated'] == 1) & (panel_aft98['filed_pat'] > 0)]
num_treated_firms_with_patent = treated_firms_with_patent['firm_id'].nunique()

print(f"total firms: {tot_firms}")
print(f"treated firms: {num_treated_firms}")
print(f"treated firms with at least one patent: {num_treated_firms_with_patent}")
print(f"control firms: {num_control_firms}")
print(f"treated share: {treated_share:.2f}%")
print(f"number of cartels: {num_cartels}")

total firms: 12880
treated firms: 147
treated firms with at least one patent: 100
control firms: 12733
treated share: 0.01%
number of cartels: 71


In [24]:
firms_aft82 = pd.read_excel('../data/orbis_data/tot_firms/tot_firms_aft82.xlsx',  sheet_name='Risultati')
panel_aft82 = panel_df.merge(firms_aft82, left_on='bvd_id', right_on='Numero BvD ID', how='right')

# summary of firms and treatment groups

tot_firms = panel_aft82['firm_id'].nunique()
num_treated_firms = panel_aft82.loc[panel_aft82['treated']==1, 'firm_id'].nunique()
num_control_firms = tot_firms - num_treated_firms
treated_share = num_treated_firms / tot_firms
num_cartels = panel_aft82['cartel_id'].nunique()
treated_firms_with_patent = panel_aft82[(panel_aft82['treated'] == 1) & (panel_aft82['filed_pat'] > 0)]
num_treated_firms_with_patent = treated_firms_with_patent['firm_id'].nunique()

print(f"total firms: {tot_firms}")
print(f"treated firms: {num_treated_firms}")
print(f"treated firms with at least one patent: {num_treated_firms_with_patent}")
print(f"control firms: {num_control_firms}")
print(f"treated share: {treated_share:.2f}%")
print(f"number of cartels: {num_cartels}")

total firms: 14591
treated firms: 168
treated firms with at least one patent: 111
control firms: 14423
treated share: 0.01%
number of cartels: 77
